# FotoApp: procesamiento de imágenes para redes sociales

**Programación Python Avanzada — Proyecto 2, primera parte**

Pruebas de las funciones de `fotoapp.py` sobre fotografías reales.

## 1. Importaciones y fotografías de entrada

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

%matplotlib inline
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFilter

ROOT = Path.cwd() if (Path.cwd() / 'fotoapp.py').is_file() else Path.cwd() / 'proyecto_2'
sys.path.insert(0, str(ROOT))
from fotoapp import publicar_foto, ecualizador, filtros, boceto

Rutas de las cinco fotografías de entrada.

In [ ]:
FOTOS = {
    'redes': ROOT / 'imagenes' / 'redes.jpg',        # adaptación a redes sociales
    'oscura': ROOT / 'imagenes' / 'oscura.jpg',      # contraste: imagen oscura
    'luminosa': ROOT / 'imagenes' / 'luminosa.jpg',  # contraste: imagen con exceso de luz
    'filtros': ROOT / 'imagenes' / 'filtrar.jpg',    # nueve filtros de Pillow
    'retrato': ROOT / 'imagenes' / 'persona.jpg',    # bocetos de línea
}

faltantes = [str(ruta) for ruta in FOTOS.values() if not ruta.is_file()]
if faltantes:
    raise FileNotFoundError('Faltan fotografías de entrada: ' + ', '.join(faltantes))

for uso, ruta in FOTOS.items():
    with Image.open(ruta) as foto:
        print(f'{uso:<9} {ruta.name:<14} {foto.width} × {foto.height} px')

SALIDA = ROOT / 'results' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
SALIDA.mkdir(parents=True)

Auxiliares para verificar los márgenes y mostrar y guardar figuras.

In [ ]:
def verificar_margenes(adaptada, resultado):
    izquierda, arriba, derecha, abajo = adaptada.info['area_contenido']
    margen = np.ones((adaptada.height, adaptada.width), dtype=bool)
    margen[arriba:abajo, izquierda:derecha] = False
    if not np.all(np.asarray(resultado)[margen] == 128):
        raise ValueError('Se modificaron los márgenes grises')


def mostrar_paneles(imagenes, titulos, nombre_archivo, columnas=None):
    columnas = columnas or len(imagenes)
    filas = -(-len(imagenes) // columnas)
    figura, ejes = plt.subplots(filas, columnas, figsize=(5 * columnas, 4.5 * filas), squeeze=False)
    for eje in ejes.flat:
        eje.axis('off')
    for eje, imagen, titulo in zip(ejes.flat, imagenes, titulos):
        eje.imshow(imagen, cmap='gray', vmin=0, vmax=255)
        eje.set_title(titulo)
    figura.tight_layout()
    figura.savefig(SALIDA / nombre_archivo, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(figura)

## 2. Adaptación a redes sociales

La foto se escala sin deformarla y se centra en un lienzo del tamaño de cada red. El espacio sobrante se completa con gris, en lugar de recortar el encuadre. Medidas: YouTube 3840 × 2160, Instagram y Facebook 1080 × 1080, Twitter/X 1200 × 675.

In [ ]:
def adaptar(red, tamano):
    adaptada = publicar_foto(str(FOTOS['redes']), red)
    ancho, alto = adaptada.info['tamano_original']
    ancho_foto, alto_foto = adaptada.info['tamano_contenido']
    izquierda, arriba, derecha, abajo = adaptada.info['area_contenido']

    if adaptada.size != tamano:
        raise ValueError('El lienzo no tiene el tamaño de la red')
    if abs(ancho_foto - round(alto_foto * ancho / alto)) > 1:
        raise ValueError('La proporción cambió')
    if abs(izquierda - (tamano[0] - derecha)) > 1 or abs(arriba - (tamano[1] - abajo)) > 1:
        raise ValueError('La foto no está centrada')
    verificar_margenes(adaptada, adaptada)

    adaptada.save(SALIDA / f'adaptada_{red.lower()}.png')
    print(f'Foto {ancho_foto} × {alto_foto} en lienzo {tamano[0]} × {tamano[1]} (escala {ancho_foto / ancho:.2f})')
    plt.figure(figsize=(6, 4))
    plt.imshow(adaptada)
    plt.title(f'{red} ({tamano[0]} × {tamano[1]})')
    plt.axis('off')
    plt.show()
    return adaptada

In [ ]:
youtube = adaptar('Youtube', (3840, 2160))

In [ ]:
instagram = adaptar('Instagram', (1080, 1080))

In [ ]:
facebook = adaptar('Facebook', (1080, 1080))

In [ ]:
twitter = adaptar('Twitter', (1200, 675))

Una escala mayor que 1 indica que la foto se amplió: conserva la proporción, pero no gana detalle.

## 3. Contraste mediante el histograma

Se ecualiza solo la luminosidad y no cada color por separado, para mejorar el contraste sin alterar los colores. Se prueba con una foto oscura y otra con exceso de luz; en cada figura se comparan los histogramas antes y después.

In [ ]:
contraste = {}


def resumen_luminosidad(imagen, area):
    y = np.asarray(imagen.crop(area).convert('YCbCr'))[:, :, 0]
    return f'media {y.mean():6.1f}   desvío {y.std():5.1f}   Y=0 {(y == 0).mean():6.2%}   Y=255 {(y == 255).mean():6.2%}'


def ecualizar(caso):
    entrada = publicar_foto(str(FOTOS[caso]), 'Instagram')
    copia = np.asarray(entrada).copy()
    resultado = ecualizador(entrada, archivo_salida=str(SALIDA / f'contraste_{caso}.png'))

    verificar_margenes(entrada, resultado)
    if not np.array_equal(np.asarray(entrada), copia):
        raise ValueError('Se modificó la imagen de entrada')
    contraste[caso] = (entrada, resultado)

    area = entrada.info['area_contenido']
    print('original  ', resumen_luminosidad(entrada, area))
    print('ecualizada', resumen_luminosidad(resultado, area))

In [ ]:
ecualizar('oscura')

In [ ]:
ecualizar('luminosa')

### Experimento adicional: ecualización por canal RGB

Para contrastar la decisión anterior, se ecualiza cada canal por separado. Este procedimiento no forma parte de `fotoapp.py`.

In [ ]:
def experimento_rgb(caso):
    entrada, ecualizada_y = contraste[caso]
    area = entrada.info['area_contenido']
    canales = [Image.fromarray(cv2.equalizeHist(np.asarray(canal))) for canal in entrada.crop(area).split()]
    ecualizada_rgb = entrada.copy()
    ecualizada_rgb.paste(Image.merge('RGB', canales), area[:2])
    verificar_margenes(entrada, ecualizada_rgb)

    mostrar_paneles([entrada, ecualizada_y, ecualizada_rgb],
                    [f'Original ({caso})', 'Ecualización de Y', 'Ecualización de R, G y B'],
                    f'experimento_rgb_{caso}.png')

In [ ]:
experimento_rgb('oscura')

In [ ]:
experimento_rgb('luminosa')

## 4. Los nueve filtros de Pillow

Los nueve filtros se aplican a la misma foto, siempre sobre la original, para comparar sus efectos sin acumularlos.

In [ ]:
entrada_filtros = publicar_foto(str(FOTOS['filtros']), 'Instagram')
copia_filtros = np.asarray(entrada_filtros).copy()
resultados_filtros = {}


def aplicar_filtro(nombre):
    archivo = SALIDA / f'filtro_{nombre.lower().replace(" ", "_")}.png'
    resultado = filtros(entrada_filtros, nombre, archivo_salida=str(archivo))
    verificar_margenes(entrada_filtros, resultado)
    if not np.array_equal(np.asarray(entrada_filtros), copia_filtros):
        raise ValueError('Los filtros se acumularon sobre la entrada')
    resultados_filtros[nombre] = resultado

In [ ]:
aplicar_filtro('Blur')

In [ ]:
aplicar_filtro('Contour')

In [ ]:
aplicar_filtro('Detail')

In [ ]:
aplicar_filtro('Edge Enhance')

In [ ]:
aplicar_filtro('Edge Enhance More')

In [ ]:
aplicar_filtro('Emboss')

In [ ]:
aplicar_filtro('Find Edges')

In [ ]:
aplicar_filtro('Sharpen')

In [ ]:
aplicar_filtro('Smooth')

La vista conjunta reúne los filtros aplicados para compararlos a la misma escala.

In [ ]:
mostrar_paneles(list(resultados_filtros.values()), list(resultados_filtros), 'filtros_vista_conjunta.png', columnas=3)

## 5. Boceto de línea a partir de un retrato

Se comparan Sobel y Canny sobre el mismo retrato, con dos configuraciones cada uno. Antes se aplica un desenfoque leve para reducir el ruido, y el resultado se invierte para obtener líneas negras sobre fondo blanco.

El porcentaje de líneas negras permite comparar configuraciones, no medir la calidad del boceto.

In [ ]:
RADIO_SUAVIZADO = 1
retrato = publicar_foto(str(FOTOS['retrato']), 'Instagram')
area = retrato.info['area_contenido']
izquierda, arriba, derecha, abajo = area
retrato_suavizado = retrato.copy()
retrato_suavizado.paste(retrato.crop(area).filter(ImageFilter.GaussianBlur(RADIO_SUAVIZADO)), area[:2])
bocetos = {}
titulos_bocetos = {}


def generar_boceto(etiqueta, metodo, umbral=0, kernel=3, umbral1=50, umbral2=150):
    archivo = SALIDA / f'boceto_{etiqueta.lower().replace(" ", "_")}.png'
    resultado = boceto(retrato_suavizado, metodo, True, umbral, kernel, umbral1, umbral2, archivo_salida=str(archivo))
    lineas = resultado[arriba:abajo, izquierda:derecha]
    if not set(np.unique(lineas)) <= {0, 255}:
        raise ValueError('El boceto no es binario')
    verificar_margenes(retrato, resultado)

    if metodo == 'Canny':
        titulo = f'Canny, umbrales {umbral1}/{umbral2}'
    else:
        titulo = f'{metodo}, umbral {umbral}'
    bocetos[etiqueta] = resultado
    titulos_bocetos[etiqueta] = titulo
    print(f'{etiqueta}: {titulo}, núcleo {kernel}. Líneas negras: {(lineas == 0).mean():.2%}')

In [ ]:
generar_boceto('Sobel A', 'Sobel', umbral=80)

In [ ]:
generar_boceto('Sobel B', 'Sobel', umbral=140)

In [ ]:
generar_boceto('Canny A', 'Canny', umbral1=50, umbral2=150)

In [ ]:
generar_boceto('Canny B', 'Canny', umbral1=100, umbral2=200)

La vista conjunta reúne las configuraciones generadas para compararlas a la misma escala.

In [ ]:
mostrar_paneles(list(bocetos.values()),
                [f'{etiqueta}: {titulo}' for etiqueta, titulo in titulos_bocetos.items()],
                'bocetos_configuraciones.png', columnas=2)

La comparación final muestra el retrato original junto a la configuración A de cada técnica.

In [ ]:
mostrar_paneles([retrato, bocetos['Sobel A'], bocetos['Canny A']],
                ['Retrato original', titulos_bocetos['Sobel A'], titulos_bocetos['Canny A']],
                'bocetos_comparacion_final.png')

## 6. Validación de entradas

Las funciones rechazan las entradas inválidas con excepciones, que el menú captura para que el usuario pueda continuar.

In [ ]:
casos_invalidos = [
    ('Red no admitida', lambda: publicar_foto(str(FOTOS['redes']), 'MySpace'), ValueError),
    ('Archivo inexistente', lambda: publicar_foto(str(ROOT / 'imagenes' / 'no_existe.jpg'), 'Instagram'), FileNotFoundError),
    ('Operación sin imagen cargada', lambda: ecualizador(None), ValueError),
    ('Filtro inexistente', lambda: filtros(entrada_filtros, 'Sepia'), ValueError),
    ('Umbrales de Canny invertidos', lambda: boceto(retrato, 'Canny', False, 0, 3, 200, 100), ValueError),
]

for descripcion, operacion, excepcion in casos_invalidos:
    try:
        operacion()
    except excepcion as error:
        print(f'{descripcion}: {type(error).__name__} — {error}')
    else:
        raise ValueError(f'No se rechazó: {descripcion}')

## 7. Conclusiones

**Adaptación.** Una única escala para ambos ejes conserva la proporción, y completar con márgenes evita recortar el encuadre. El costo son las franjas grises cuando la relación de aspecto de la foto difiere de la de la red. Registrar el área de la foto permite excluir esas franjas de todas las operaciones posteriores sin depender del color del fondo, que podría confundirse con zonas grises de la propia imagen.

**Contraste.** La ecualización redistribuye los niveles de luminosidad para ocupar todo el rango. El aumento del desvío de Y indica una distribución más amplia, no una imagen necesariamente mejor: en las zonas uniformes también puede aumentar el ruido. Los píxeles que ya estaban en 0 o en 255 no recuperan el detalle perdido, y la ecualización tampoco lo reconstruye. Ecualizar solo Y conserva los colores mejor que ecualizar R, G y B por separado, ya que este último procedimiento modifica la relación entre canales.

**Filtros.** Los filtros de suavizado y de realce actúan sobre la textura y la nitidez, pero conservan la fotografía reconocible. Contour, Find Edges y Emboss la transforman en una representación de bordes o de relieve. La magnitud del cambio que produce un filtro no indica su calidad: la elección depende del uso que se le quiera dar a la imagen.

**Bocetos.** En Sobel, el umbral regula directamente la cantidad de líneas. Canny tiende a producir bordes más finos y continuos, a cambio de dos umbrales que hay que ajustar en conjunto. Las densidades impresas permiten comparar configuraciones, pero la valoración del parecido requiere observar las figuras.

**Alcance.** Las observaciones visuales dependen de las fotografías de entrada y deben hacerse sobre las figuras de cada ejecución. Este notebook no realiza descargas: la carga desde una URL está cubierta por las pruebas automatizadas de `test_fotoapp.py`.